Filen bygger en RAG chatbot som är specialiserad på hudvård med tips och rekommendationer från kicks.se. Den innehåller metadata kring 3 olika hudyper (fet, torr och kombinerad) och har färdiga dokument kring produkter och tips anpassade efter slutanvändarens input från en Streamlit-app. Chatboten får endast ge rekommendationer och tips från inladdade dokument och använder sedan OpenAIs LLM för att chatta med slutanvändaren. Alla txt-dokument som laddas in delas in i hudtyp och kategorier utifrån sina filnamn, varpå det är viktigt att följa namnstandard när txt-filer döps.
Nedan specificeras vilka tillägg som används till vad:

langchain: Själva "ramverket" som binder ihop textfiler från kicks.se med AI-modellen.

langchain-openai: Den specifika bryggan för att prata med OpenAIs modeller.

chromadb: En Vector Database. Det är här alla meningar sparas som sökbara vektorer (siffror).

In [1]:
# En kod för att rensa tidigare inläsningar om något skall uppdateras, för att undvika att det krockar med gammal data
base_folder = "." # Hänvisar till den lokala mappen
all_documents = [] # Skapar en tom lista med alla dokument som laddas in

In [3]:
# Alla imports som behövs för att köra samtliga kodblock
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:

# Skapar en tom behållare för att samla in all textdata innan den bearbetas
base_folder = "." 
all_documents = [] 

# Ladda in alla txt-filer och används filnamnen för att tagga filerna så att RAG enkelt kan hitta
for root, dirs, files in os.walk(base_folder):
    for filename in files:
        if filename.endswith(".txt"):
            file_path = os.path.join(root, filename) # här skapas sökväg baserat på filnamn
            
            try: # Testar att läsa in filer med UTF-8-kodning, och hoppar vidare till nästa fil om någon fil är skadad utan att krasha
                loader = TextLoader(file_path, encoding='utf-8')
                loaded_docs = loader.load()
                
                for doc in loaded_docs: # Skapar variabler med små bokstäver så att inga ord missas pga stora/små bokstäver i filnamnet
                    file_lower = filename.lower()
                    path_lower = file_path.lower()
                    
                    # Delar in i olika hudtyper och sparar som metadata utifrån filnamn
                    if "torr-hud" in path_lower or "torr-hud" in file_lower:
                        doc.metadata["skin_type"] = "torr-hud"
                    elif "fet-hud" in path_lower or "fet-hud" in file_lower:
                        doc.metadata["skin_type"] = "fet-hud"
                    elif "kombinerad-hud" in path_lower or "kombinerad-hud" in file_lower:
                        doc.metadata["skin_type"] = "kombinerad-hud"
                    else:
                        doc.metadata["skin_type"] = "allmänt"
                    
                    # Delar upp i kategorier ch sparar som metadata utifrån filnamn
                    if "rengöring" in file_lower or "rengoring" in file_lower:
                        doc.metadata["category"] = "rengöring"
                    elif "serum" in file_lower:
                        doc.metadata["category"] = "serum"
                    elif "ansiktskräm" in file_lower or "ansiktskram" in file_lower or "lotion" in file_lower:
                        doc.metadata["category"] = "ansiktskräm"
                    else:
                        doc.metadata["category"] = "allmänt"
                    
                    # Sparar källan för spårbarhet
                    doc.metadata["source"] = filename
                    all_documents.append(doc)
            except Exception as e:
                print(f"Kunde inte ladda {filename}: {e}") # Felhantering om filen inte kan laddas in och sparas enligt ovan

# Delar upp txt-dokumenten i chunks. Recurive delar upp texten vid naturliga tecken som radbrytningar eller punkter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_chunks = text_splitter.split_documents(all_documents)

print(f"KLART! Totalt antal chunks skapade: {len(final_chunks)}")

KLART! Totalt antal chunks skapade: 50


Ovan ser korrekt ut med 49 skapade chunks. Nu går jag vidare med att skapa embeddings som "fångar" kontexten av alla chunks genom att omvandla alla textstycken (chunks) till matematiska vektorer (listor med siffror).

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Initierar embeddings och använder nyckel mot OpenAIs Embedding modell
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

# Skapar ett FAISS-index av alla chunks
vectorstore = FAISS.from_documents(
    documents=final_chunks,
    embedding=embeddings
)

# Sparar index lokalt
vectorstore.save_local("faiss_skincare_v1")

print(f"Nytt FAISS-index skapat med {len(final_chunks)} taggade textbitar!")

# Testa att söka efter rengöring för fet hud
filtered_results = vectorstore.similarity_search(
    "Vilken tvätt passar mig?",
    k=10,
    filter={
        "$and": [
            {"skin_type": "fet-hud"},
            {"category": "rengöring"}
        ]
    }
)

print("\n--- Resultat av test (Endast rengöring för fet hud) ---")
for r in filtered_results[:2]:
    print(f"Källa: {r.metadata['source']}")
    print(f"Innehåll: {r.page_content[:150]}...")
    print("-" * 10)

Nytt FAISS-index skapat med 50 taggade textbitar!

--- Resultat av test (Endast rengöring för fet hud) ---
Källa: rengöring-fet-hud.txt
Innehåll: Pris: 280 kr.

Länk: https://www.kicks.se/kiehls-rare-earth-deep-pore-daily-cleanser-150-ml

2. COSRX - Low pH Good Morning Gel Cleanser

Typ: Mild re...
----------


Skapar själva Hjärnan (LLM-kedjan) som pratar med databasen och sammanfattar svaren åt användaren.

In [15]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

# TEST
user_question = "Jag vill ha hjälp med hudvård."
selected_skin_type = "fet-hud"
selected_category = "ansiktskräm"

llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

system_prompt = (
    "Du är en hudvårdsexpert på Kicks. "
    "Använd endast informationen i kontexten för att svara på frågan. "
    "VIKTIGT: Om samma produkt förekommer flera gånger i kontexten, presentera den bara EN gång. "
    "Om ingen produkt matchar filtren exakt, säg det tydligt och föreslå då något närliggande alternativ från kontexten. "
    "Hitta inte på produkter, priser eller länkar."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

vectorstore = FAISS.load_local(
    "faiss_skincare_v1",
    embeddings,
    allow_dangerous_deserialization=True
)

# Hämta brett först
candidate_docs = vectorstore.similarity_search(
    user_question,
    k=20
)

# Filtrera strikt själv i Python
strict_docs = [
    doc for doc in candidate_docs
    if doc.metadata.get("skin_type") == selected_skin_type
    and doc.metadata.get("category") == selected_category
]

# Om inget exakt matchar: försök med bara hudtyp
fallback_docs = [
    doc for doc in candidate_docs
    if doc.metadata.get("skin_type") == selected_skin_type
]

if strict_docs:
    final_docs = strict_docs[:5]
    mode_text = f"Exakt filtermatch: {selected_skin_type} + {selected_category}"
elif fallback_docs:
    final_docs = fallback_docs[:5]
    mode_text = f"Ingen exakt {selected_category}; fallback till {selected_skin_type}"
else:
    final_docs = []
    mode_text = "Ingen relevant kontext hittades"

context = "\n\n".join(doc.page_content for doc in final_docs)

messages = prompt.invoke({
    "input": user_question,
    "context": context if context else "Ingen relevant kontext hittades."
})

response = llm.invoke(messages)

print("\n--- TEST-INFO ---")
print(mode_text)
print(f"Antal träffar efter manuell filtrering: {len(final_docs)}")

print("\n--- HÄMTADE DOKUMENT ---")
for i, doc in enumerate(final_docs, 1):
    print(f"\nDokument {i}")
    print("Metadata:", doc.metadata)
    print("Innehåll:", doc.page_content[:250])

print("\n--- CHATBOTENS SVAR ---")
print(response.content)


--- TEST-INFO ---
Exakt filtermatch: fet-hud + ansiktskräm
Antal träffar efter manuell filtrering: 1

--- HÄMTADE DOKUMENT ---

Dokument 1
Metadata: {'source': 'ansiktskräm-fet-hud.txt', 'skin_type': 'fet-hud', 'category': 'ansiktskräm'}
Innehåll: 2. La Roche-Posay - Effaclar Mat

Typ: Mattgörande ansiktskräm.

Varför den passar: Motverkar glans och hjälper till att dra ihop porerna. Ger en matt finish som håller hela dagen.

Pris: 219 kr.

Länk: https://www.kicks.se/la-roche-posay-effaclar-ma

--- CHATBOTENS SVAR ---
Självklart! Om du letar efter en produkt som motverkar glans och hjälper till att dra ihop porerna, kan jag rekommendera "La Roche-Posay - Effaclar Mat". Denna mattgörande ansiktskräm ger en matt finish som håller hela dagen och kostar 219 kr. Om du har specifika önskemål eller behov, kan jag hjälpa till att föreslå andra produkter baserade på vad du letar efter.


Jag är nöjd med ovan output för torr hud, och vill nu gå vidare med även andra hudtyper.